In [28]:
import os
import json
import shutil
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from dtw import dtw
import pandas as pd

from collections import Counter



In [29]:
all_data = []

pose_indices = [0, 15, 16, 17, 18, 19, 20]
hand_indices = [0, 4, 7, 8, 11, 12, 15, 16, 19, 20]

In [30]:
def load_landmarks(filenames, num_frames, glossIndex = 4):
    
    
    # Prepare storage for data and labels
    video_data = []
    labels = []
    
    for file in filenames:
     
        # print(file)
        gloss = file.split('/')[glossIndex]
        # print(gloss)


        with open(file, 'rb') as f:
                landmarks = pickle.load(f)
                    
                # Prepare frame storage for each video with fixed number of frames
                frames = []

                # Get frame sampling step (skip frames if necessary)
                total_frames = len(landmarks)
                step = max(1, total_frames // num_frames)
                    
                # Process frames
                for i in range(num_frames):
                    frame_index = i * step if total_frames >= num_frames else i
                    
                    if frame_index < total_frames:
                        frame = landmarks[frame_index]
                    else:
                        frame = None
                        
                    # Extract and flatten required points
                    pose_points = frame['pose_landmarks'] if frame and frame['pose_landmarks'] else [None] * 33
                    left_hand_points = frame['left_hand_landmarks'] if frame and frame['left_hand_landmarks'] else [None] * 21
                    right_hand_points = frame['right_hand_landmarks'] if frame and frame['right_hand_landmarks'] else [None] * 21
                        
                    # Collect only specified indices and flatten them
                    extracted_points = []

                    # Collect pose points first
                    for idx in pose_indices:
                        point = pose_points[idx] if pose_points[idx] is not None else {'x': 0, 'y': 0}
                        extracted_points.extend([point['x'], point['y']])
                        
                    # print(left_hand_points)
                    for idx in hand_indices:
                        point = left_hand_points[idx] if left_hand_points[idx] is not None else {'x': 0, 'y': 0}
                        extracted_points.extend([point['x'], point['y']])

                    # Collect right hand points
                    for idx in hand_indices:
                        point = right_hand_points[idx] if right_hand_points[idx] is not None else {'x': 0, 'y': 0}
                        extracted_points.extend([point['x'], point['y']])
                        
                    frames.append(extracted_points)
                    
                # Add the processed video frames to the main array
                video_data.append(frames)
                labels.append(gloss)
                

    # Convert to numpy arrays
    video_data = np.array(video_data)
    labels = np.array(labels)
    
    return video_data, labels

In [31]:


# Assume load_landmarks is already defined as in your existing code
# It takes file_names, num_frames, and optional glossIndex, returning video_data and labels

# Parameters
num_frames = 30  # Fixed number of frames per video, as in your example

# --- Step 1: Load the First Dataset ---
data_directory1 = '../all_outputs/output_ssl_small/'
file_names1 = []
for root, _, files in os.walk(data_directory1):
    for filename in files:
        file_names1.append(os.path.join(root, filename))
print(f"First dataset: {len(file_names1)} files found")
video_data1, labels1 = load_landmarks(file_names1, num_frames, glossIndex=3)

# --- Step 2: Load the Second Dataset ---
data_directory2 = '../all_outputs/output_include/'
file_names2 = []
for root, _, files in os.walk(data_directory2):
    for filename in files:
        file_names2.append(os.path.join(root, filename))
print(f"Second dataset: {len(file_names2)} files found")
video_data2, labels2 = load_landmarks(file_names2, num_frames, glossIndex=4)


First dataset: 1280 files found
Second dataset: 4022 files found


In [38]:
# --- Step 3: Aggregate Videos by Gloss Using Median Sequence ---
def get_median_sequence(videos):
    """
    Compute the median sequence across multiple videos for a gloss.
    
    Args:
        videos (list of np.array): List of video sequences, each with shape [30, num_features].
                                  Each sequence represents skeletal data over 30 frames.
    
    Returns:
        np.array: Median sequence with shape [30, num_features], representing the "typical" movement.
    """
    # Stack all videos for this gloss into a single array
    # Shape becomes [num_instances, 30, num_features]
    stacked_videos = np.stack(videos, axis=0)
    # Compute the median across the instance axis (axis=0)
    # This gives a single sequence that captures the central tendency of the gloss's movements
    median_sequence = np.median(stacked_videos, axis=0)  # Shape: [30, num_features]
    return median_sequence

def aggregate_by_gloss(video_data, labels, file_names):
    """
    Aggregate videos by gloss, computing the median sequence and selecting a representative file.
    
    Args:
        video_data (np.array): Shape [num_videos, 30, num_features], skeletal data for all videos.
        labels (np.array): Shape [num_videos], gloss labels for each video.
        file_names (list): List of file paths corresponding to each video.
    
    Returns:
        dict: Mapping from gloss to {'sequence': median_sequence, 'file': representative_file}.
              Each entry represents a gloss with its median sequence and a reference file.
    """
    # Initialize a dictionary to group videos and files by gloss
    gloss_to_data = {}
    for i, gloss in enumerate(labels):
        if gloss not in gloss_to_data:
            gloss_to_data[gloss] = {'videos': [], 'files': []}
        gloss_to_data[gloss]['videos'].append(video_data[i])
        gloss_to_data[gloss]['files'].append(file_names[i])
    
    # Process each gloss to compute its median sequence and assign a representative file
    gloss_reps = {}
    for gloss, data in gloss_to_data.items():
        # Compute the median sequence for this gloss
        median_seq = get_median_sequence(data['videos'])
        # Choose the first file as the representative (for reference in output)
        # Note: Since we use the median sequence, the file choice doesn’t affect DTW
        rep_file = data['files'][0]
        gloss_reps[gloss] = {'sequence': median_seq, 'file': rep_file}
    return gloss_reps

In [39]:
# Apply aggregation to both datasets
# Assuming video_data1, labels1, file_names1, video_data2, labels2, file_names2 are provided
gloss_reps1 = aggregate_by_gloss(video_data1, labels1, file_names1)
gloss_reps2 = aggregate_by_gloss(video_data2, labels2, file_names2)
print(f"First dataset: {len(gloss_reps1)} unique glosses")
print(f"Second dataset: {len(gloss_reps2)} unique glosses")

# --- Step 4: Compute DTW for Gloss Pairs Using Median Sequences ---
similarity_metrics = []

# Calculate total number of gloss pairs for progress tracking
total_pairs = len(gloss_reps1) * len(gloss_reps2)
current_pair = 0

# Compare each gloss from dataset1 with each gloss from dataset2
for gloss1 in gloss_reps1:
    for gloss2 in gloss_reps2:
        current_pair += 1
        print(f"Processing pair {current_pair}/{total_pairs}: gloss '{gloss1}' vs gloss '{gloss2}'")
        
        # Get the median sequences for the current gloss pair
        seq1 = gloss_reps1[gloss1]['sequence']  # Shape: [30, num_features]
        seq2 = gloss_reps2[gloss2]['sequence']  # Shape: [30, num_features]
        
        # Compute DTW distance between the two sequences
        # DTW aligns sequences temporally and measures similarity using Euclidean distance
        alignment = dtw(seq1, seq2, keep_internals=True)
        distance = alignment.distance  # Lower distance indicates more similar movements
        
        # Store the similarity result
        similarity_metrics.append({
            'gloss1': gloss1,
            'gloss2': gloss2,
            'video1': gloss_reps1[gloss1]['file'],
            'video2': gloss_reps2[gloss2]['file'],
            'dtw_distance': distance
        })

# --- Step 5: Sort and Save Results ---
# Convert the list of similarity metrics to a DataFrame for easy handling
df = pd.DataFrame(similarity_metrics)

# Sort by DTW distance (ascending) to list most similar pairs first
df_sorted = df.sort_values(by='dtw_distance', ascending=True)

# Save to CSV is commented out as per request, but print a confirmation message
# df_sorted.to_csv('similarity_metrics_gloss_pairs.csv', index=False)
print(f"Saved {len(similarity_metrics)} similarity metrics to 'similarity_metrics_gloss_pairs.csv'")
print(f"Results sorted by DTW distance (lowest = most similar)")

First dataset: 64 unique glosses
Second dataset: 242 unique glosses
Processing pair 1/15488: gloss 'To_You' vs gloss 'Saturday'
Processing pair 2/15488: gloss 'To_You' vs gloss 'Yesterday'
Processing pair 3/15488: gloss 'To_You' vs gloss 'Tomorrow'
Processing pair 4/15488: gloss 'To_You' vs gloss 'Night'
Processing pair 5/15488: gloss 'To_You' vs gloss 'Second'
Processing pair 6/15488: gloss 'To_You' vs gloss 'Friday'
Processing pair 7/15488: gloss 'To_You' vs gloss 'Evening'
Processing pair 8/15488: gloss 'To_You' vs gloss 'Sunday'
Processing pair 9/15488: gloss 'To_You' vs gloss 'Week'
Processing pair 10/15488: gloss 'To_You' vs gloss 'Year'
Processing pair 11/15488: gloss 'To_You' vs gloss 'Hour'
Processing pair 12/15488: gloss 'To_You' vs gloss 'Tuesday'
Processing pair 13/15488: gloss 'To_You' vs gloss 'Minute'
Processing pair 14/15488: gloss 'To_You' vs gloss 'Wednesday'
Processing pair 15/15488: gloss 'To_You' vs gloss 'Today'
Processing pair 16/15488: gloss 'To_You' vs gloss 'M

In [40]:
df_sorted[:30]

,gloss1,gloss2,video1,video2,dtw_distance
7787,Make,Page,../all_outputs/output_ssl_small/Make/2-4-w0382...,../all_outputs/output_include/Home/Page/MVI_49...,78.872387
7763,Make,Month,../all_outputs/output_ssl_small/Make/2-4-w0382...,../all_outputs/output_include/Days_and_Time/Mo...,79.459846
14539,My,Month,../all_outputs/output_ssl_small/My/4-5-w030202...,../all_outputs/output_include/Days_and_Time/Mo...,80.142203
7757,Make,Wednesday,../all_outputs/output_ssl_small/Make/2-4-w0382...,../all_outputs/output_include/Days_and_Time/We...,80.658566
14533,My,Wednesday,../all_outputs/output_ssl_small/My/4-5-w030202...,../all_outputs/output_include/Days_and_Time/We...,80.676626
6553,w064,Month,../all_outputs/output_ssl_small/w064/3-2-w0642...,../all_outputs/output_include/Days_and_Time/Mo...,80.742142
7037,See,Month,../all_outputs/output_ssl_small/See/3-2-w04020...,../all_outputs/output_include/Days_and_Time/Mo...,81.995315
6547,w064,Wednesday,../all_outputs/output_ssl_small/w064/3-2-w0642...,../all_outputs/output_include/Days_and_Time/We...,82.740395
14586,My,Box,../all_outputs/output_ssl_small/My/4-5-w030202...,../all_outputs/output_include/Home/Box/MVI_907...,83.130867
14563,My,Page,../all_outputs/output_ssl_small/My/4-5-w030202...,../all_outputs/output_include/Home/Page/MVI_49...,83.391976


In [43]:
file_name = "similarity_metrics_gloss_pairs"

In [ ]:
df_sorted.to_csv(file_name, index=False)

In [45]:
import pandas as pd

# --- Step 1: Read the CSV File ---
# Load the similarity metrics CSV generated by the Aggregated DTW method
# Expected columns: gloss1, gloss2, video1, video2, dtw_distance
csv_file = file_name
try:
    df = pd.read_csv(csv_file)
except FileNotFoundError:
    print(f"Error: CSV file '{csv_file}' not found.")
    exit(1)

# --- Step 2: Find Most Similar Include Word for Each SSL Word ---
# Group by gloss1 (ssl word) and find the row with the minimum dtw_distance
# This identifies the gloss2 (include word) with the most similar movement
best_matches = df.loc[df.groupby('gloss1')['dtw_distance'].idxmin()]

# --- Step 3: Output Results ---
# Print a header for clarity
print("Most Similar Include Words for Each SSL Word:")
print("---------------------------------------------")

# Iterate through the best matches and print details
for _, row in best_matches.iterrows():
    print(f"SSL Word: {row['gloss1']}")
    print(f"Most Similar Include Word: {row['gloss2']}")
    print(f"Include Video Path: {row['video2']}")
    print(f"Similarity Score (DTW Distance): {row['dtw_distance']:.4f}")
    print("---------------------------------------------")

# Optionally, save the results to a new CSV for reference
# This creates a reduced dataset mapping each ssl gloss to its best include match
output_csv = 'best_matches_ssl_to_include.csv'
best_matches[['gloss1', 'gloss2', 'video2', 'dtw_distance']].to_csv(output_csv, index=False)
print(f"Saved best matches to '{output_csv}'")

Error: CSV file 'similarity_metrics_gloss_pairs' not found.
Most Similar Include Words for Each SSL Word:
---------------------------------------------
SSL Word: Again
Most Similar Include Word: Store or Shop
Include Video Path: ../all_outputs/output_include/Places/Store or Shop/MVI_3555.pkl
Similarity Score (DTW Distance): 126.9797
---------------------------------------------
SSL Word: Age
Most Similar Include Word: Month
Include Video Path: ../all_outputs/output_include/Days_and_Time/Month/MVI_5487.pkl
Similarity Score (DTW Distance): 87.4053
---------------------------------------------
SSL Word: Big
Most Similar Include Word: Month
Include Video Path: ../all_outputs/output_include/Days_and_Time/Month/MVI_5487.pkl
Similarity Score (DTW Distance): 85.9214
---------------------------------------------
SSL Word: Can?
Most Similar Include Word: Month
Include Video Path: ../all_outputs/output_include/Days_and_Time/Month/MVI_5487.pkl
Similarity Score (DTW Distance): 118.6738
------------